# 02 Sanity Check WRDS Extracts

This notebook is a lightweight research-facing driver for `sanity_check_wrds_extract.py`.

The production logic lives in the `.py` script so the same checks can be called from `01_run_wrds_extracts.ipynb`, the terminal, or future pipeline automation. Use this notebook when you want to rerun QA interactively and inspect the generated tables.

## Run sanity checks

This validates the raw membership, CRSP panel, FF3, and SPY benchmark extract files, then saves CSV/TXT/PNG outputs under `data/sanity_checks`.

In [ ]:
%run ./sanity_check_wrds_extract.py --project-root . --raw-dir data/raw --benchmark-dir data/benchmark --outdir data/sanity_checks

## Inspect saved outputs

These cells read the artifacts generated by the script. They are intentionally separate from the check logic so the script stays the single source of truth.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

outdir = Path("data/sanity_checks")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [ ]:
print((outdir / "coverage_summary.txt").read_text())

In [ ]:
for filename in [
    "extract_dataset_summary.csv",
    "final_checks.csv",
    "missingness_summary.csv",
    "duplicate_key_summary.csv",
    "factor_coverage_summary.csv",
]:
    print(filename)
    display(pd.read_csv(outdir / filename))

## Monthly count diagnostics

The count tables are often the fastest way to spot extraction coverage issues before building the modeling panel.

In [ ]:
counts = pd.read_csv(outdir / "crsp_monthly_counts.csv", parse_dates=["month_end"])

display(counts.head())
display(counts.tail())
display(counts[["membership_count", "crsp_count", "crsp_minus_membership"]].describe())

In [ ]:
largest_count_gaps = counts.assign(
    abs_gap=counts["crsp_minus_membership"].abs()
).sort_values(["abs_gap", "month_end"], ascending=[False, True])

display(largest_count_gaps.head(20))